# 📄 hermes-acp-sdk — figures for the paper

> **📸 For the paper figure:** Menu **Run ▸ Run All Cells**, then screenshot each
> output cell marked **📸**. Use the **light** theme (Settings ▸ Theme ▸ JupyterLab Light)
> and zoom the browser to **~175%** (`Cmd`/`Ctrl` `+`) so the fonts are crisp. Crop to
> roughly **9 cm** wide (one card) for a two-column paper.

In [ ]:
import logging
for _n in ("mcp", "httpx", "uvicorn", "uvicorn.error", "uvicorn.access", "asyncio"):
    logging.getLogger(_n).setLevel(logging.WARNING)   # keep the output clean

from collections import Counter
from hermes_acp_sdk import HermesClient, AgentText, AgentThought, Usage, Finished
print("ready — run the two cells below and screenshot their output")

## 📸 Figure 1 (best) — the whole API: screenshot the **code AND output**

In [ ]:
from hermes_acp_sdk import HermesClient, AgentThought, AgentText, Usage

# This is the entire API. Driving Hermes over ACP would otherwise mean writing a
# 12-method JSON-RPC client; the SDK turns it into two `async with` + one `async for`.
async with HermesClient() as agent:              # spawns `hermes acp`, does the ACP handshake
    async with agent.session() as chat:          # selects a model for you
        reasoning = answer = ""
        async for event in chat.prompt("In one sentence, what is a Python traceback?"):
            if isinstance(event, AgentThought):
                reasoning += event.text           # the agent's private reasoning
            elif isinstance(event, AgentText):
                answer += event.text              # the answer, streamed token by token
            elif isinstance(event, Usage):
                tokens = event.total_tokens       # real token accounting

print("🧠 reasoning:", reasoning[:80].strip(), "…")
print("💬 answer   :", answer.strip())
print("📊 tokens   :", tokens)

## 📸 Figure 2 — the typed event stream in detail (screenshot the output)

In [ ]:
from collections import Counter

events = []
async with HermesClient() as agent:
    async with agent.session() as chat:
        async for event in chat.prompt("Think it through, then answer: what is 12 * 12?"):
            events.append(event)

# Every raw ACP notification becomes a typed Python object you can match on:
print("events received:", dict(Counter(type(e).__name__ for e in events)))
print("AgentText  ->", "".join(e.text for e in events if isinstance(e, AgentText)).strip())
print("Usage      ->", next((e for e in events if isinstance(e, Usage)), None))

## 📸 Figure 3 — isolated profile via clone_provider (real evidence, no API key handled)

In [ ]:
import subprocess

# Create the app's OWN profile, cloning the host's provider. Note: the code never
# references an API key — clone_provider carries the credentials into the profile.
async with HermesClient(profile="paper-demo", clone_provider=True, auto_prefix=True) as agent:
    async with agent.session() as chat:
        async for _ in chat.prompt("Reply: OK"):
            pass

# Proof from Hermes itself: the app now has its own profile beside the host's default.
print(subprocess.run(["hermes", "profile", "list"], capture_output=True, text=True).stdout)

---
**Captions**

*Figure 1 — the entire hermes-acp-sdk API: two `async with` and one `async for` drive the
Hermes agent over the Agent Client Protocol from any Python app (no Jupyter), yielding a
typed event stream — the agent's reasoning, its answer, and real token usage. Driving ACP
directly would instead require implementing a 12-method JSON-RPC client.*

*Figure 2 — every raw ACP notification is surfaced as a typed Python object (`AgentThought`,
`AgentText`, `Usage`, …) the app can match on, instead of hand-parsing protocol JSON.*

*Figure 3 — `clone_provider=True` gives the app its own isolated Hermes profile (separate
memory, sessions, skills) while inheriting the host's provider, so the app drives a
fully-configured agent without ever handling an API key.*